# 🏠 House Price Prediction — EDA & Model Training

This notebook walks through the full machine learning workflow step by step:

1. Load & explore the dataset
2. Visualise distributions and correlations
3. Preprocess the data
4. Train and compare two regression models
5. Evaluate model performance
6. Inspect feature importances
7. Save the model for use in the Flask API

**Dataset:** California Housing (built into scikit-learn — no download needed)

## 1. Import Libraries

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # allow imports from project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries loaded successfully!')

## 2. Load the Dataset

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 3. Dataset Overview

In [ ]:
print('--- Data Types & Non-Null Counts ---')
df.info()

In [ ]:
print('--- Missing Values ---')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

In [ ]:
print('--- Statistical Summary ---')
df.describe().round(2)

## 4. Exploratory Data Analysis (EDA)

### 4.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Median House Value')
axes[0].set_xlabel('Median House Value ($100,000s)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['MedHouseVal'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Box Plot of Median House Value')
axes[1].set_ylabel('Median House Value ($100,000s)')

plt.tight_layout()
plt.show()
print(f"Mean: ${df['MedHouseVal'].mean()*100_000:,.0f}")
print(f"Median: ${df['MedHouseVal'].median()*100_000:,.0f}")

### 4.2 Feature Distributions

In [ ]:
features = housing.feature_names
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=40, color='teal', edgecolor='white', alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4.3 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, square=True
)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()
print('\nTop correlations with MedHouseVal:')
print(corr['MedHouseVal'].sort_values(ascending=False).to_string())

### 4.4 Scatter Plots: Key Features vs House Price

In [ ]:
top_features = ['MedInc', 'AveRooms', 'HouseAge', 'Latitude']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, feat in zip(axes, top_features):
    ax.scatter(df[feat], df['MedHouseVal'], alpha=0.1, s=5, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('MedHouseVal')
    ax.set_title(f'{feat} vs Price')

plt.suptitle('Feature vs House Price Scatter Plots', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Preprocessing

In [ ]:
X = df[housing.feature_names].values
y = df['MedHouseVal'].values

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (zero mean, unit variance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {X_train_scaled.shape[0]}')
print(f'Test samples     : {X_test_scaled.shape[0]}')
print(f'Features         : {X_train_scaled.shape[1]}')

## 6. Train Models

In [ ]:
# ── Baseline: Linear Regression ────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
print('Linear Regression trained.')

# ── Main model: Random Forest ───────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
print('Random Forest trained.')

## 7. Model Evaluation

In [ ]:
def get_metrics(model, X, y):
    y_pred = model.predict(X)
    return {
        'MAE':  mean_absolute_error(y, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y, y_pred)),
        'R2':   r2_score(y, y_pred),
    }

metrics = pd.DataFrame({
    'Linear Regression': get_metrics(lr, X_test_scaled, y_test),
    'Random Forest':     get_metrics(rf, X_test_scaled, y_test),
}).T.round(4)

print('\n=== Model Comparison (Test Set) ===')
print(metrics.to_string())
print('\nNote: MAE and RMSE are in units of $100,000.')
print(f"      Random Forest MAE in dollars: ${metrics.loc['Random Forest','MAE']*100_000:,.0f}")

### 7.1 Actual vs Predicted Plot

In [ ]:
y_pred_rf = rf.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name, color in zip(
    axes,
    [lr, rf],
    ['Linear Regression', 'Random Forest'],
    ['coral', 'steelblue']
):
    y_pred = model.predict(X_test_scaled)
    ax.scatter(y_test, y_pred, alpha=0.2, s=5, color=color)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()], 'k--', lw=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Price ($100k)')
    ax.set_ylabel('Predicted Price ($100k)')
    ax.set_title(f'{name}\nR² = {r2_score(y_test, y_pred):.3f}')
    ax.legend()

plt.suptitle('Actual vs Predicted House Prices', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Feature Importances (Random Forest)

In [ ]:
fi = pd.DataFrame({
    'Feature':    housing.feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(fi['Feature'], fi['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Random Forest — Feature Importances')
plt.tight_layout()
plt.show()

print('\nFeature Importances (sorted):')
print(fi.sort_values('Importance', ascending=False).to_string(index=False))

## 9. Save Model & Scaler

In [ ]:
os.makedirs('../model', exist_ok=True)
joblib.dump(rf,     '../model/house_price_model.pkl')
joblib.dump(scaler, '../model/scaler.pkl')
print('[OK] model/house_price_model.pkl saved')
print('[OK] model/scaler.pkl saved')

## 10. Test a Single Prediction

In [ ]:
sample = np.array([[8.3252, 41.0, 6.984, 1.024, 322.0, 2.555, 37.88, -122.23]])
sample_scaled = scaler.transform(sample)
predicted = rf.predict(sample_scaled)[0]
print(f'Sample input  : {sample[0].tolist()}')
print(f'Predicted price: ${predicted * 100_000:,.0f}')